# Phase 3: Rolling AR Forecast, No Adoption

Goal: run the null market, fit a rolling AR forecast offline against realised returns, and confirm that the rule recovers positive out-of-sample R2 from the phi term before any trader adopts it.

Reference: section 3.4 of the proposal for the rolling AR rule and section 4.1 for MSFE and out-of-sample R2.

**Note on realised vs demand-adjusted return.** This phase has zero adoption, so no trader trades on the forecast. Aggregate adopter demand contributes nothing to the contemporaneous price impact, and the realised return `r_{t+1}` and the demand-adjusted return `x_{t+1} = r_{t+1} - mu * D_t` differ only by the small effect of null orders on `D_t`. For this offline forecasting sanity check the two targets are effectively aligned, so reporting realised-return R2 alone here is appropriate. The realised vs demand-adjusted distinction becomes important from phase 4 onward, when adopters start trading and the two targets diverge. Note that the demand-adjusted return is a counterfactual that strips out contemporaneous price impact only; it is not the full regression residual, which would be `sigma * epsilon = r - phi*r_lag - mu*D`.

In [ ]:
# Number of traders.
N = 200

# Number of periods.
T = 8000

# Market-maker price-impact coefficient (equation 5).
mu = 0.05

# Reduced-form AR(1) coefficient in the return law (equation 6).
phi = 0.25

# Standard deviation of exogenous news shocks.
sigma_news = 0.01

# Standard deviation of individual null orders (equation 9).
sigma_q = 1.0

# Rolling AR order.
ar_order = 1

# Rolling estimation windows for the AR rule.
ar_windows = [50, 100, 250]

# Trailing window for rolling forecast-performance metrics.
eval_window = 1000

# Random seed.
seed = 43

# Output paths.
fig_dir = "../results/figures"
data_dir = "../results/data"

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import matplotlib.pyplot as plt

from reflexive_market import forecast, simulate
from reflexive_market.metrics import msfe, rolling_msfe, rolling_oos_r2, sign_accuracy

In [ ]:
rng = np.random.default_rng(seed)
ar_windows = np.array(ar_windows, dtype=int)

In [ ]:
result = simulate.run(
    T=T,
    N=N,
    mu=mu,
    phi=phi,
    sigma_news=sigma_news,
    sigma_q=sigma_q,
    rng=rng,
)

prices = result["prices"]
returns = result["returns"]
demand = result["demand"]

In [ ]:
forecasts_by_window = {}
params_by_window = {}
rolling_msfe_by_window = {}
rolling_r2_by_window = {}
summary = np.zeros((len(ar_windows), 5))

for i, window in enumerate(ar_windows):
    window = int(window)
    forecasts, params = forecast.rolling_ar_forecast(returns, window, p=ar_order)
    r_msfe = rolling_msfe(returns, forecasts, eval_window)
    r2 = rolling_oos_r2(returns, forecasts, eval_window)

    forecasts_by_window[window] = forecasts
    params_by_window[window] = params
    rolling_msfe_by_window[window] = r_msfe
    rolling_r2_by_window[window] = r2
    summary[i] = np.array([
        window,
        msfe(returns, forecasts),
        np.nanmean(r2),
        np.nanmean(params[:, 1]),
        sign_accuracy(returns, forecasts),
    ])

In [ ]:
print(f"input phi:              {phi:+.4f}")
print("window   MSFE       mean rolling R2   mean phi_hat   sign accuracy")
for row in summary:
    print(f"{int(row[0]):>6}   {row[1]:.8f}   {row[2]:+.4f}            {row[3]:+.4f}        {row[4]:.4f}")

## Rolling AR coefficient

The AR(1) coefficient estimates should fluctuate around the input phi. Shorter windows react faster but have visibly higher variance.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for window in ar_windows:
    window = int(window)
    ax.plot(params_by_window[window][:, 1], linewidth=0.8, label=f"w = {window}")

ax.axhline(phi, color="k", linewidth=0.8, linestyle="--", label="input phi")
ax.set_title("Rolling AR(1) coefficient estimates")
ax.set_xlabel("Time period")
ax.set_ylabel("Estimated phi")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_03_ar_coefficients.png", dpi=150)
plt.show()

## Rolling forecast error

Rolling MSFE should be stable because no trader is using the forecast yet. Later phases can compare forecast erosion against this no-adoption benchmark.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for window in ar_windows:
    window = int(window)
    ax.plot(rolling_msfe_by_window[window], linewidth=0.9, label=f"w = {window}")

ax.set_title("Rolling mean squared forecast error")
ax.set_xlabel("Time period")
ax.set_ylabel("MSFE")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_03_rolling_msfe.png", dpi=150)
plt.show()

## Rolling out-of-sample R2

The forecast should retain positive out-of-sample R2 against a constant-mean benchmark, because this phase still contains the reduced-form phi term and no adoption feedback.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.5))
for window in ar_windows:
    window = int(window)
    ax.plot(rolling_r2_by_window[window], linewidth=0.9, label=f"w = {window}")

ax.axhline(0.0, color="k", linewidth=0.8, linestyle="--")
ax.set_title("Rolling out-of-sample R2")
ax.set_xlabel("Time period")
ax.set_ylabel("OOS R2")
ax.legend()
fig.tight_layout()
fig.savefig(f"{fig_dir}/phase_03_rolling_oos_r2.png", dpi=150)
plt.show()

## Save numeric summary

Stored as a small npz with the per-window summary table and the run inputs. The simulator state and per-period forecast/parameter traces are reproducible from the seed; saving them inflates the file without adding diagnostic value beyond rerunning the notebook.

In [ ]:
np.savez(
    f"{data_dir}/phase_03_ar_forecast.npz",
    summary=summary,
    ar_windows=ar_windows,
    ar_order=np.array(ar_order),
    eval_window=np.array(eval_window),
    phi_input=np.array(phi),
    seed=np.array(seed),
)

## Done when

- Rolling out-of-sample R2 is positive after warm-up.
- Rolling MSFE is stable because no trader uses the forecast yet.
- Longer windows give lower-variance estimates of the AR coefficient.